# Windows 로컬 eTL RAG 실습 노트북

이 노트북은 `data_etl/` 문서를 사용해서 RAG 파이프라인을 실행하는 Windows 로컬 실습용 노트북입니다.

패키지 설치 셀은 넣지 않았습니다. 이미 프로젝트 실행 환경이 준비되어 있다고 가정합니다.

각 단계는 다음 방식으로 실행합니다.

1. 바로 위 Python 셀에서 `QUERY`, `TOP_K`, `LIMIT` 같은 값을 바꿉니다.
2. 바로 아래 `%%bash` 셀을 실행합니다.
3. 결과는 cell output에 출력됩니다.


## 1. 프로젝트 폴더 경로 설정

`PROJECT_DIR`만 본인 컴퓨터의 프로젝트 폴더 경로로 고치세요.

Windows 경로 앞에는 `r`을 붙입니다.


In [ ]:
from pathlib import Path
import os
import sys

# 예: r"C:\sct-project-graphrag"
PROJECT_DIR = Path(r"C:\sct-project-graphrag")

REQUIRED_MARKERS = ["pyproject.toml", "scripts_etl", "data_etl"]

if not PROJECT_DIR.exists():
    raise FileNotFoundError(f"프로젝트 폴더를 찾지 못했습니다: {PROJECT_DIR}")

missing = [marker for marker in REQUIRED_MARKERS if not (PROJECT_DIR / marker).exists()]
if missing:
    raise RuntimeError(
        f"프로젝트 폴더로 보이지 않습니다: {PROJECT_DIR}\n"
        f"없는 항목: {missing}\n"
        "zip이 한 겹 더 들어가서 풀렸는지 확인하고 PROJECT_DIR을 안쪽 폴더로 고치세요."
    )

os.chdir(PROJECT_DIR)
os.environ["PROJECT_DIR"] = str(PROJECT_DIR)

print("프로젝트 폴더:", PROJECT_DIR)
print("현재 작업 폴더:", Path.cwd())
print("노트북 Python:", sys.executable)


## 2. API 환경변수 설정

LLM을 호출하는 단계에서 사용합니다.

raw BM25 검색만 실행할 때는 API key가 없어도 됩니다.


In [ ]:
import getpass
import os

api_key = getpass.getpass("조별 API key를 입력하세요. raw 검색만 할 경우 Enter를 눌러 건너뛰어도 됩니다: ")

if api_key:
    os.environ["OPENAI_API_KEY"] = api_key

os.environ["BASE_URL"] = "ldi.snu.ac.kr:30000"
os.environ["OPENAI_MODEL"] = "gpt-5.4-mini"
os.environ["OPENAI_REASONING_EFFORT"] = "low"
os.environ["OPENAI_EMBEDDING_MODEL"] = "text-embedding-3-small"

print("환경변수 설정 완료")
print("BASE_URL:", os.environ["BASE_URL"])
print("OPENAI_MODEL:", os.environ["OPENAI_MODEL"])


## 3. 공통 파일 경로

필요하면 여기에서 `data_etl`을 다른 데이터 폴더로 바꾸면 됩니다.


In [ ]:
import os

DATA_DIR = "data_etl"

RAW_PAGES = f"{DATA_DIR}/processed/pdf_pages.jsonl"
TOPICS = f"{DATA_DIR}/processed/guide_topics.jsonl"
TOPIC_INDEX = f"{DATA_DIR}/indexes/topic_embedding_index.npz"
TOPIC_METADATA = f"{DATA_DIR}/indexes/topic_embedding_metadata.json"
TOPIC_FEATURES = f"{DATA_DIR}/processed/topic_features.jsonl"
TOPIC_GRAPH = f"{DATA_DIR}/indexes/topic_graph_with_similarity.json"

for key, value in {
    "RAW_PAGES": RAW_PAGES,
    "TOPICS": TOPICS,
    "TOPIC_INDEX": TOPIC_INDEX,
    "TOPIC_METADATA": TOPIC_METADATA,
    "TOPIC_FEATURES": TOPIC_FEATURES,
    "TOPIC_GRAPH": TOPIC_GRAPH,
}.items():
    os.environ[key] = value

print("RAW_PAGES:", RAW_PAGES)
print("TOPICS:", TOPICS)


## 4. Raw text BM25 검색

원문 텍스트에서 바로 키워드 검색을 합니다.

API를 사용하지 않습니다.


In [ ]:
import os

QUERY = "과제를 만들고 제출 기간을 설정하려면 어디에서 무엇을 해야 하나?"
TOP_K = 3

os.environ["QUERY"] = QUERY
os.environ["TOP_K"] = str(TOP_K)

print("QUERY:", QUERY)
print("TOP_K:", TOP_K)


In [ ]:
%%bash
cd "$PROJECT_DIR"
python scripts_etl/06_bm25_raw_text_demo.py \
  "$QUERY" \
  --input "$RAW_PAGES" \
  --top-k "$TOP_K"


## 5. Raw text BM25 RAG 답변

원문 검색 결과를 LLM에게 넣어 답변을 생성합니다.

API를 사용합니다.


In [ ]:
import os

QUERY = "동영상 출결을 확인하려면 어디에서 어떤 정보를 봐야 하나?"
TOP_K = 5
MAX_CONTEXT_CHARS = 7000

os.environ["QUERY"] = QUERY
os.environ["TOP_K"] = str(TOP_K)
os.environ["MAX_CONTEXT_CHARS"] = str(MAX_CONTEXT_CHARS)

print("QUERY:", QUERY)
print("TOP_K:", TOP_K)


In [ ]:
%%bash
cd "$PROJECT_DIR"
python scripts_etl/09_bm25_raw_rag_answer.py \
  "$QUERY" \
  --input "$RAW_PAGES" \
  --top-k "$TOP_K" \
  --max-context-chars "$MAX_CONTEXT_CHARS"


## 6. eTL topic 구조화 추출

원문 문서를 읽고, 사용자가 물어볼 만한 질문과 절차를 구조화합니다.

API를 사용합니다.

처음 테스트할 때는 `LIMIT`을 작게 두는 것이 좋습니다.


In [ ]:
import os

LIMIT = 5
CONCURRENCY = 2

os.environ["LIMIT"] = str(LIMIT)
os.environ["CONCURRENCY"] = str(CONCURRENCY)

print("LIMIT:", LIMIT)
print("CONCURRENCY:", CONCURRENCY)
print("OUTPUT:", TOPICS)


In [ ]:
%%bash
cd "$PROJECT_DIR"
python scripts_etl/04_extract_guide_topics.py \
  "$RAW_PAGES" \
  "$TOPICS" \
  --limit "$LIMIT" \
  --concurrency "$CONCURRENCY"


## 7. Topic BM25 검색

04번에서 구조화한 topic을 검색합니다.

API를 사용하지 않습니다.


In [ ]:
import os

QUERY = "과제 제출 기간과 파일 제출 설정"
TOP_K = 5

os.environ["QUERY"] = QUERY
os.environ["TOP_K"] = str(TOP_K)

print("QUERY:", QUERY)
print("TOP_K:", TOP_K)


In [ ]:
%%bash
cd "$PROJECT_DIR"
python scripts_etl/07_bm25_topic_demo.py \
  "$QUERY" \
  --input "$TOPICS" \
  --top-k "$TOP_K"


## 8. Topic embedding index 생성

구조화된 topic을 embedding index로 만듭니다.

API를 사용합니다.


In [ ]:
import os

LIMIT = 50
BATCH_SIZE = 64

os.environ["LIMIT"] = str(LIMIT)
os.environ["BATCH_SIZE"] = str(BATCH_SIZE)

print("LIMIT:", LIMIT)
print("BATCH_SIZE:", BATCH_SIZE)
print("INDEX:", TOPIC_INDEX)


In [ ]:
%%bash
cd "$PROJECT_DIR"
python scripts_etl/05_build_topic_embedding_index.py \
  --input "$TOPICS" \
  --index "$TOPIC_INDEX" \
  --metadata "$TOPIC_METADATA" \
  --limit "$LIMIT" \
  --batch-size "$BATCH_SIZE"


## 9. Dense topic 검색

질문을 embedding으로 바꾸고, topic embedding index에서 의미 검색을 합니다.

API를 사용합니다.


In [ ]:
import os

QUERY = "학생들이 제출한 과제를 확인하고 성적을 처리하는 방법"
TOP_K = 5

os.environ["QUERY"] = QUERY
os.environ["TOP_K"] = str(TOP_K)

print("QUERY:", QUERY)
print("TOP_K:", TOP_K)


In [ ]:
%%bash
cd "$PROJECT_DIR"
python scripts_etl/08_dense_topic_search.py \
  "$QUERY" \
  --index "$TOPIC_INDEX" \
  --metadata "$TOPIC_METADATA" \
  --top-k "$TOP_K"


## 10. Topic BM25 + raw text RAG 답변

구조화 topic 검색 결과와 원문 context를 함께 사용해 답변합니다.

API를 사용합니다.


In [ ]:
import os

QUERY = "과제를 만들고 제출 현황을 확인한 다음 성적 처리는 어디에서 해야 하나?"
TOP_K = 4
RAW_CONTEXT = "same-document"

os.environ["QUERY"] = QUERY
os.environ["TOP_K"] = str(TOP_K)
os.environ["RAW_CONTEXT"] = RAW_CONTEXT

print("QUERY:", QUERY)
print("TOP_K:", TOP_K)
print("RAW_CONTEXT:", RAW_CONTEXT)


In [ ]:
%%bash
cd "$PROJECT_DIR"
python scripts_etl/10_bm25_topic_rag_answer.py \
  "$QUERY" \
  --topics "$TOPICS" \
  --pages "$RAW_PAGES" \
  --top-k "$TOP_K" \
  --raw-context "$RAW_CONTEXT"


## 11. Search agent demo

agent가 필요한 검색어를 스스로 고르고, 여러 번 검색한 뒤 답변합니다.

API를 사용합니다.


In [ ]:
import os

QUERY = "과제를 새로 만들고 제출 기간과 파일 제출을 설정한 뒤, 학생 제출 현황이나 성적 처리는 어디에서 확인해야 하나?"
TOP_K = 4
MAX_STEPS = 4

os.environ["QUERY"] = QUERY
os.environ["TOP_K"] = str(TOP_K)
os.environ["MAX_STEPS"] = str(MAX_STEPS)

print("QUERY:", QUERY)
print("TOP_K:", TOP_K)
print("MAX_STEPS:", MAX_STEPS)


In [ ]:
%%bash
cd "$PROJECT_DIR"
python scripts_etl/15_search_agent_demo.py \
  "$QUERY" \
  --topics "$TOPICS" \
  --pages "$RAW_PAGES" \
  --top-k "$TOP_K" \
  --max-steps "$MAX_STEPS"


## 12. Graph feature 추출

Graph RAG까지 볼 때만 실행합니다.

API를 사용합니다.


In [ ]:
import os

LIMIT = 30
CONCURRENCY = 2

os.environ["LIMIT"] = str(LIMIT)
os.environ["CONCURRENCY"] = str(CONCURRENCY)

print("LIMIT:", LIMIT)
print("OUTPUT:", TOPIC_FEATURES)


In [ ]:
%%bash
cd "$PROJECT_DIR"
python scripts_etl/11_extract_graph_features.py \
  --input "$TOPICS" \
  --output "$TOPIC_FEATURES" \
  --limit "$LIMIT" \
  --concurrency "$CONCURRENCY"


## 13. Topic graph 생성

11번 결과를 graph JSON으로 만듭니다.

`ADD_SIMILARITY`가 `0`이면 API를 사용하지 않습니다. `1`이면 similarity edge 생성을 위해 embedding API를 사용합니다.


In [ ]:
import os

ADD_SIMILARITY = 0

os.environ["ADD_SIMILARITY"] = str(ADD_SIMILARITY)

print("ADD_SIMILARITY:", ADD_SIMILARITY)
print("GRAPH:", TOPIC_GRAPH)


In [ ]:
%%bash
cd "$PROJECT_DIR"
if [ "$ADD_SIMILARITY" = "1" ]; then
  python scripts_etl/12_build_topic_graph.py \
    --input "$TOPIC_FEATURES" \
    --output "$TOPIC_GRAPH" \
    --add-similarity
else
  python scripts_etl/12_build_topic_graph.py \
    --input "$TOPIC_FEATURES" \
    --output "$TOPIC_GRAPH"
fi


## 14. Graph viewer HTML 생성

브라우저에서 볼 수 있는 graph HTML 파일을 만듭니다.

API를 사용하지 않습니다.


In [ ]:
import os

VIEWER_OUTPUT = "results/etl_graph_viewer_sigma.html"
os.environ["VIEWER_OUTPUT"] = VIEWER_OUTPUT

print("VIEWER_OUTPUT:", VIEWER_OUTPUT)


In [ ]:
%%bash
cd "$PROJECT_DIR"
python scripts_etl/13_export_sigma_viewer.py \
  --input "$TOPIC_GRAPH" \
  --output "$VIEWER_OUTPUT"


## 15. Graph RAG 답변

Graph에서 관련 topic을 찾고 LLM으로 답변합니다.

API를 사용합니다.


In [ ]:
import os

QUERY = "과제 생성, 제출 설정, 성적 처리와 관련된 화면과 설정은 어떻게 연결되어 있나?"
TOP_SEEDS = 8
MAX_TOPICS = 30

os.environ["QUERY"] = QUERY
os.environ["TOP_SEEDS"] = str(TOP_SEEDS)
os.environ["MAX_TOPICS"] = str(MAX_TOPICS)

print("QUERY:", QUERY)
print("TOP_SEEDS:", TOP_SEEDS)
print("MAX_TOPICS:", MAX_TOPICS)


In [ ]:
%%bash
cd "$PROJECT_DIR"
python scripts_etl/14_graph_rag_answer.py \
  "$QUERY" \
  --graph "$TOPIC_GRAPH" \
  --top-seeds "$TOP_SEEDS" \
  --max-topics "$MAX_TOPICS"
